# Défi quotidien : Pipelines LangChain avec LLM open-source (Étudiant)
Utilisez ce notebook guidé avec des TODOs. Fonctionne sur CPU avec de petits modèles HF (par exemple, flan-t5-small).

## Ce que vous apprendrez
- Configurer LangChain avec des modèles open-source légers.
- Construire une chaîne LLM en utilisant un modèle de prompt.
- Composer un pipeline exécutable en deux étapes (résumé ? puces).
- Bonus : ajouter une chaîne de conversation simple avec mémoire.

## Ce que vous allez créer
- Environnement installé pour LangChain + transformers.
- Chaîne LLM qui réécrit le texte dans un style plus simple.
- Pipeline exécutable qui résume puis met en liste des puces le texte.
- (Bonus) Chaîne de conversation montrant la mémoire.

## Partie 1 : Configuration de l'environnement (rapide)
Installez les paquets nécessaires. Le CPU est suffisant pour les petits modèles.

In [ ]:

# TODO: verify hardware (optional)
# !nvidia-smi || echo "CPU runtime"


In [ ]:

# TODO: install dependencies
pip install "transformers==4.37.2" "langchain==0.1.7" "langchain-community==0.0.20" "langchain-core==0.1.23"

## Partie 2 : Charger un petit modèle et construire votre première chaîne LLM
Utilisez un petit modèle (par exemple, google/flan-t5-small) pour que l'inférence reste rapide.

In [ ]:

# TODO: import libs
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from langchain_community.llms import HuggingFacePipeline
from langchain import PromptTemplate, LLMChain


In [ ]:

# TODO: choose a small model
model_name = "google/flan-t5-small"  # keep small for CPU


In [ ]:

# TODO: load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


In [ ]:

# TODO: create a generation pipeline
gen_pipeline = pipeline(
    task="text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=128,
)
llm = HuggingFacePipeline(pipeline=gen_pipeline)


In [ ]:

# TODO: build prompt + LLMChain for friendly rewriting
template = "Rewrite this text to be simpler for beginners:{text}"
prompt = PromptTemplate(template=template, input_variables=["text"])
chain = LLMChain(prompt=prompt, llm=llm)

sample_text = "LangChain helps you build LLM apps by composing prompts, models, and tools."
rewritten = chain.run(text=sample_text)
print(rewritten)


## Partie 3 : Pipeline en deux étapes (résumé ? puces)
Résumez un paragraphe, puis transformez-le en 3 puces en utilisant le même LLM.

In [3]:
from langchain_core.runnables import RunnableLambda  # if needed, depending on version
from langchain_core.prompts import PromptTemplate

#To-Do define run templates
summary_prompt = PromptTemplate(
    template="Summarize the following paragraph:\n\n{paragraph}",
    input_variables=["paragraph"],
)
bullets_prompt = PromptTemplate(
    template="Convert the following summary into 3 bullet points:\n\n{summary}",
    input_variables=["summary"],
)

In [ ]:
# First stage: paragraph -> summary (string)
summary_chain = summary_prompt | llm

# Full chain:
# 1. Take input {"paragraph": ...}
# 2. Run summary_chain to get a summary string
# 3. Wrap into {"summary": summary}
# 4. Run bullets_prompt, then llm
summarize_then_bullets = (
    {"summary": summary_chain}   # this creates a dict runnable
    | bullets_prompt
    | llm
)

In [ ]:
paragraph = """LangChain is a framework for building applications with large language models by composing prompts, models, and tools. It supports chains, agents, and retrieval workflows."""
bullets_output = summarize_then_bullets.invoke({"paragraph": paragraph})
print(bullets_output)


## Partie 4 (Bonus) : Chaîne de conversation avec mémoire
Montrez comment deux tours conservent le contexte.

In [ ]:

# TODO: build a simple conversation chain
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferMemory

memory = ConversationBufferMemory()
convo = ConversationChain(llm=llm, memory=memory, verbose=False)

reply1 = convo.predict(input="Hi there! What's LangChain?")
reply2 = convo.predict(input="Can it help me build a simple chatbot?")
print("Turn 1:", reply1)
print("Turn 2:", reply2)


## Vos observations (à remplir)
- Latence : À FAIRE
- Qualité : À FAIRE
- Particularités : À FAIRE (par exemple, hallucinations, réponses courtes)